Kaggle/Drive

In [ ]:
!pip install kagglehub --quiet
import kagglehub
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


Import and store dataset

In [ ]:
# Mount Google Drive (if not already done)
from google.colab import drive
drive.mount('/content/drive')

# Install Kaggle CLI if missing
!pip install -q kaggle

# Setup Kaggle API credentials (make sure kaggle.json is in your Drive)
!mkdir -p ~/.kaggle
!cp /content/drive/MyDrive/kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

# Download and unzip dataset locally
!kaggle datasets download -d tanjemahamed/mental-fatigue-level-detection-fatigueset-data --unzip -p /content/fatigue_data

# List local downloaded files to verify
!ls /content/fatigue_data

import shutil
import os

source_dir = '/content/fatigue_data/fatigueset'  # The actual dataset folder
drive_dest = '/content/drive/MyDrive/Fatigue_Set'

# Create destination folder if it doesn't exist
os.makedirs(drive_dest, exist_ok=True)

# Define full destination path for the folder copy
dest_dir = os.path.join(drive_dest, 'fatigueset')

# Remove destination folder if it exists to avoid copytree error
if os.path.exists(dest_dir):
    shutil.rmtree(dest_dir)

# Recursively copy entire directory
shutil.copytree(source_dir, dest_dir)

print(f'Dataset folder copied recursively to: {dest_dir}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Dataset URL: https://www.kaggle.com/datasets/tanjemahamed/mental-fatigue-level-detection-fatigueset-data
License(s): MIT
 99% 614M/618M [00:02<00:00, 130MB/s]
100% 618M/618M [00:02<00:00, 248MB/s]
fatigueset
Dataset folder copied recursively to: /content/drive/MyDrive/Fatigue_Set/fatigueset


Focused files and column structure

In [ ]:
import os
import pandas as pd

BASE_PATH = '/content/drive/MyDrive/Fatigue_Set/fatigueset'
persons = [f'{i:02d}' for i in range(1, 13)]
sessions = [f'{i:02d}' for i in range(1, 4)]

# Selected sensor files relevant for your fatigue detection model
selected_sensor_files = [
    'wrist_hr.csv',
    'wrist_ibi.csv',
    'wrist_acc.csv',
    'wrist_eda.csv',
    'wrist_skin_temperature.csv',
    'exp_fatigue.csv'
]

for person in persons:
    for session in sessions:
        session_folder = os.path.join(BASE_PATH, person, session)
        print(f'\nPerson: {person}, Session: {session}')
        for sensor_file in selected_sensor_files:
            file_path = os.path.join(session_folder, sensor_file)
            if os.path.exists(file_path):
                try:
                    df = pd.read_csv(file_path, nrows=3)  # Read only first few rows
                    print(f'{sensor_file} columns: {list(df.columns)}')
                except Exception as e:
                    print(f"Error reading {file_path}: {e}")
            else:
                print(f'{sensor_file} not found in session {session} of person {person}')



Person: 01, Session: 01
wrist_hr.csv columns: ['timestamp', 'hr']
wrist_ibi.csv columns: ['timestamp', 'duration']
wrist_acc.csv columns: ['timestamp', 'ax', 'ay', 'az']
wrist_eda.csv columns: ['timestamp', 'eda']
wrist_skin_temperature.csv columns: ['timestamp', 'temp']
exp_fatigue.csv columns: ['measurementNumber', 'physicalFatigueAnswerTime', 'mentalFatigueAnswerTime', 'fatigueSurveySubmissionTime', 'physicalFatigueScore', 'mentalFatigueScore']

Person: 01, Session: 02
wrist_hr.csv columns: ['timestamp', 'hr']
wrist_ibi.csv columns: ['timestamp', 'duration']
wrist_acc.csv columns: ['timestamp', 'ax', 'ay', 'az']
wrist_eda.csv columns: ['timestamp', 'eda']
wrist_skin_temperature.csv columns: ['timestamp', 'temp']
exp_fatigue.csv columns: ['measurementNumber', 'physicalFatigueAnswerTime', 'mentalFatigueAnswerTime', 'fatigueSurveySubmissionTime', 'physicalFatigueScore', 'mentalFatigueScore']

Person: 01, Session: 03
wrist_hr.csv columns: ['timestamp', 'hr']
wrist_ibi.csv columns: ['ti

Focused columns sample


In [ ]:
import os
import pandas as pd
import numpy as np

BASE_PATH = '/content/drive/MyDrive/Fatigue_Set/fatigueset'
PERSONS = [f'{i:02d}' for i in range(1, 13)]
SESSIONS = [f'{i:02d}' for i in range(1, 4)]

SENSOR_FILES = {
    'wrist_hr.csv': ['hr'],
    'wrist_ibi.csv': ['duration'],
    'wrist_acc.csv': ['ax', 'ay', 'az'],
    'wrist_eda.csv': ['eda'],
    'wrist_skin_temperature.csv': ['temp']
}

WINDOW_SIZE_SEC = 30
STEP_SIZE_SEC = 15

def get_min_sampling_interval(filepath):
    df = pd.read_csv(filepath)
    if 'timestamp' not in df.columns or df.empty:
        return None
    ts = pd.to_datetime(df['timestamp'], unit='ms')
    intervals = ts.diff().dropna()
    min_interval = intervals.min()
    return min_interval

def load_and_resample(filepath, cols, resample_freq):
    df = pd.read_csv(filepath)
    df = df.dropna(subset=['timestamp'])
    df['timestamp'] = pd.to_datetime(df['timestamp'], unit='ms')
    df.set_index('timestamp', inplace=True)

    original_counts = df[cols].count()

    df_resampled = df[cols].resample(resample_freq).mean().interpolate()
    resampled_counts = df_resampled.count()

    print(f"File: {os.path.basename(filepath)}")
    for col in cols:
        orig = original_counts[col]
        resampled = resampled_counts[col]
        percent = (resampled / orig * 100) if orig > 0 else 0
        print(f"  Column: {col}, Original entries: {orig}, Resampled entries: {resampled}, Percentage: {percent:.2f}%")
    return df_resampled

def load_and_merge_session(person, session):
    session_path = os.path.join(BASE_PATH, person, session)
    sensor_min_intervals = []

    for file_name in SENSOR_FILES.keys():
        file_path = os.path.join(session_path, file_name)
        if os.path.exists(file_path):
            min_intv = get_min_sampling_interval(file_path)
            if min_intv is not None:
                sensor_min_intervals.append(min_intv)
    if not sensor_min_intervals:
        print(f"No sensor data found for person {person} session {session}")
        return None

    best_interval = max(sensor_min_intervals)
    resample_milliseconds = int(best_interval.total_seconds() * 1000)
    resample_freq_str = f"{resample_milliseconds}ms"
    print(f"\nPerson {person}, Session {session}, Resampling freq chosen: every {resample_freq_str}")

    data_frames = []
    for file_name, cols in SENSOR_FILES.items():
        file_path = os.path.join(session_path, file_name)
        if os.path.exists(file_path):
            df_resampled = load_and_resample(file_path, cols, resample_freq_str)
            data_frames.append(df_resampled)
    if not data_frames:
        return None

    merged_df = pd.concat(data_frames, axis=1).interpolate().dropna()
    print(f"\nSynchronized merged data sample for Person {person} Session {session}:")
    print(merged_df.head(20))

    return merged_df

# Run for all persons and sessions
for person in PERSONS:
    for session in SESSIONS:
        print(f"Processing Person {person}, Session {session}")
        merged_data = load_and_merge_session(person, session)


Processing Person 01, Session 01

Person 01, Session 01, Resampling freq chosen: every 1000ms
File: wrist_hr.csv
  Column: hr, Original entries: 1579, Resampled entries: 1579, Percentage: 100.00%
File: wrist_ibi.csv
  Column: duration, Original entries: 900, Resampled entries: 1530, Percentage: 170.00%
File: wrist_acc.csv
  Column: ax, Original entries: 50497, Resampled entries: 1579, Percentage: 3.13%
  Column: ay, Original entries: 50497, Resampled entries: 1579, Percentage: 3.13%
  Column: az, Original entries: 50497, Resampled entries: 1579, Percentage: 3.13%
File: wrist_eda.csv
  Column: eda, Original entries: 6313, Resampled entries: 1579, Percentage: 25.01%
File: wrist_skin_temperature.csv
  Column: temp, Original entries: 6313, Resampled entries: 1579, Percentage: 25.01%

Synchronized merged data sample for Person 01 Session 01:
                        hr     duration        ax        ay        az  \
timestamp                                                               
2021-

Features , Labels , Count data

In [ ]:
import os
import pandas as pd
import numpy as np

BASE_PATH = '/content/drive/MyDrive/Fatigue_Set/fatigueset'
PERSONS   = [f'{i:02d}' for i in range(1, 13)]
SESSIONS  = [f'{i:02d}' for i in range(1, 4)]

SENSOR_FILES = {
    'wrist_hr.csv': ['hr'],
    'wrist_ibi.csv': ['duration'],
    'wrist_acc.csv': ['ax', 'ay', 'az'],
    'wrist_eda.csv': ['eda'],
    'wrist_skin_temperature.csv': ['temp']
}

WINDOW_SIZE_SEC = 30
STEP_SIZE_SEC   = 15

session_type_map = {
    '01': 0,  # baseline
    '02': 1,  # physical
    '03': 2   # mental
}

# ========== Utilities for Data Processing ==========

def get_min_sampling_interval(filepath):
    df = pd.read_csv(filepath)
    if 'timestamp' not in df.columns or df.empty:
        return None
    ts = pd.to_datetime(df['timestamp'], unit='ms')
    intervals = ts.diff().dropna()
    return intervals.min()

def load_and_resample(filepath, cols, resample_freq):
    df = pd.read_csv(filepath)
    df = df.dropna(subset=['timestamp'])
    df['timestamp'] = pd.to_datetime(df['timestamp'], unit='ms')
    df.set_index('timestamp', inplace=True)

    df_resampled = df[cols].resample(resample_freq).mean().interpolate()
    return df_resampled

def load_and_merge_session(person, session):
    session_path = os.path.join(BASE_PATH, person, session)
    sensor_min_intervals = []
    for file_name in SENSOR_FILES.keys():
        file_path = os.path.join(session_path, file_name)
        if os.path.exists(file_path):
            min_intv = get_min_sampling_interval(file_path)
            if min_intv is not None:
                sensor_min_intervals.append(min_intv)
    if not sensor_min_intervals:
        return None, None, None

    best_interval = max(sensor_min_intervals)
    resample_milliseconds = int(best_interval.total_seconds() * 1000)
    resample_freq_str = f"{resample_milliseconds}ms"

    data_frames = []
    for file_name, cols in SENSOR_FILES.items():
        file_path = os.path.join(session_path, file_name)
        if os.path.exists(file_path):
            df_resampled = load_and_resample(file_path, cols, resample_freq_str)
            data_frames.append(df_resampled)
    if not data_frames:
        return None, None, None

    merged_df = pd.concat(data_frames, axis=1).interpolate().dropna()
    return merged_df, None, None

def windowed_segmentation(data, window_size_sec=30, step_size_sec=15, fs_hz=None):
    if fs_hz is None:
        timedelta = (data.index[1] - data.index[0]).total_seconds()
        fs_hz = 1 / timedelta

    window_size_samples = int(window_size_sec * fs_hz)
    step_size_samples   = int(step_size_sec * fs_hz)

    segments, indices = [], []
    for start in range(0, len(data) - window_size_samples + 1, step_size_samples):
        end = start + window_size_samples
        segment = data.iloc[start:end]
        segments.append(segment)
        indices.append(segment.index[0])
    return segments, indices

def extract_features(segment):
    features = {}
    for col in segment.columns:
        features[f'{col}_mean'] = segment[col].mean()
        features[f'{col}_std']  = segment[col].std()
    return features

def load_fatigue_labels(person, session, session_start_timestamp):
    fatigue_path = os.path.join(BASE_PATH, person, session, 'exp_fatigue.csv')
    if not os.path.exists(fatigue_path):
        return None
    df = pd.read_csv(fatigue_path)
    df['fatigueSurveySubmissionDatetime'] = df['fatigueSurveySubmissionTime'].apply(
        lambda x: pd.Timestamp(session_start_timestamp) + pd.Timedelta(seconds=x)
    )
    return df

def align_labels_to_windows_time_based(label_df, window_starts):
    """
    Instead of nearest label assignment, interpolate fatigue scores over time.
    - Assumes label_df has 'fatigueSurveySubmissionDatetime',
      'physicalFatigueScore', 'mentalFatigueScore'
    - window_starts is a list/array of pandas Timestamps
    """
    submission_times = pd.to_datetime(label_df['fatigueSurveySubmissionDatetime'])
    phys_scores = label_df['physicalFatigueScore'].values
    ment_scores = label_df['mentalFatigueScore'].values

    # Build interpolation functions (time → fatigue score)
    phys_interp = np.interp(
        [ts.value for ts in window_starts],     # convert to ns int
        [t.value for t in submission_times],
        phys_scores
    )
    ment_interp = np.interp(
        [ts.value for ts in window_starts],
        [t.value for t in submission_times],
        ment_scores
    )

    labels = []
    for p, m in zip(phys_interp, ment_interp):
        labels.append({
            'physicalFatigueScore': p,
            'mentalFatigueScore': m
        })
    return labels


def process_person_session(person, session):
    merged_df, _, _ = load_and_merge_session(person, session)
    if merged_df is None:
        return None

    ts_deltas = merged_df.index.to_series().diff().dropna()
    fs_hz = 1 / ts_deltas.mean().total_seconds()

    windows, window_starts = windowed_segmentation(merged_df, WINDOW_SIZE_SEC, STEP_SIZE_SEC, fs_hz)

    feature_list = [extract_features(window) for window in windows]
    features_df = pd.DataFrame(feature_list)

    session_start_timestamp = merged_df.index.min()
    fatigue_labels_df = load_fatigue_labels(person, session, session_start_timestamp)
    if fatigue_labels_df is None:
        return None

    labels = align_labels_to_windows_time_based(fatigue_labels_df, window_starts)
    labels_df = pd.DataFrame(labels)

    merged_features_labels_df = pd.concat([features_df.reset_index(drop=True),
                                           labels_df.reset_index(drop=True)], axis=1)

    merged_features_labels_df['window_start'] = pd.to_datetime(window_starts).values
    merged_features_labels_df['person'] = person
    merged_features_labels_df['session'] = session
    return merged_features_labels_df

# ========== Build Whole Dataset ==========

all_data = []
for person in PERSONS:
    for session in SESSIONS:
        df = process_person_session(person, session)
        if df is not None:
            all_data.append(df)

final_df = pd.concat(all_data, ignore_index=True)
print("Final dataframe shape:", final_df.shape)

save_path = '/content/drive/MyDrive/Fatigue_Set/final_feature_label_dataset_interpolated.csv'
final_df.to_csv(save_path, index=False)
print(f"Saved merged dataset to {save_path}")


Final dataframe shape: (2819, 19)
Saved merged dataset to /content/drive/MyDrive/Fatigue_Set/final_feature_label_dataset_interpolated.csv


In [ ]:
ffld=pd.read_csv('/content/drive/MyDrive/Fatigue_Set/final_feature_label_dataset_interpolated.csv')
ffld.columns

Index(['hr_mean', 'hr_std', 'duration_mean', 'duration_std', 'ax_mean',
       'ax_std', 'ay_mean', 'ay_std', 'az_mean', 'az_std', 'eda_mean',
       'eda_std', 'temp_mean', 'temp_std', 'physicalFatigueScore',
       'mentalFatigueScore', 'window_start', 'person', 'session'],
      dtype='object')

Data Normalization

In [ ]:
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
import joblib
import random

# Load dataset
df = pd.read_csv('/content/drive/MyDrive/Fatigue_Set/final_feature_label_dataset_interpolated.csv')

# Sensor feature columns
feature_cols = [
    'hr_mean', 'hr_std', 'duration_mean', 'duration_std',
    'ax_mean', 'ax_std', 'ay_mean', 'ay_std', 'az_mean', 'az_std',
    'eda_mean', 'eda_std', 'temp_mean', 'temp_std'
]

# Target label columns
label_cols = ['physicalFatigueScore', 'mentalFatigueScore']

# ---- Hold-out split BEFORE scaling ----
all_pairs = df.groupby(['person','session']).size().index.tolist()
random.seed(42); random.shuffle(all_pairs)
train_pairs, test_pairs = all_pairs[:31], all_pairs[31:]

train_df = pd.concat([df[(df.person==p) & (df.session==s)] for (p,s) in train_pairs])
test_df  = pd.concat([df[(df.person==p) & (df.session==s)] for (p,s) in test_pairs])

# ---- Fit scaler only on training data ----
feat_scaler = MinMaxScaler()
train_df[feature_cols] = feat_scaler.fit_transform(train_df[feature_cols])
test_df[feature_cols]  = feat_scaler.transform(test_df[feature_cols])
joblib.dump(feat_scaler, 'feature_scaler.save')

# ---- Scale labels (optional: usually for regression stability) ----
label_scaler = MinMaxScaler()
train_df[label_cols] = label_scaler.fit_transform(train_df[label_cols])
test_df[label_cols]  = label_scaler.transform(test_df[label_cols])
joblib.dump(label_scaler, 'label_scaler.save')

# ---- Save normalized train/test sets ----
train_df.to_csv('/content/drive/MyDrive/Fatigue_Set/final_train_normalized.csv', index=False)
test_df.to_csv('/content/drive/MyDrive/Fatigue_Set/final_test_normalized.csv', index=False)

print("Features + Labels normalized (train/test separately) and saved.")

Features + Labels normalized (train/test separately) and saved.


In [ ]:
df=pd.read_csv('/content/drive/MyDrive/Fatigue_Set/final_train_normalized.csv')
df.head()

,hr_mean,hr_std,duration_mean,duration_std,ax_mean,ax_std,ay_mean,ay_std,az_mean,az_std,eda_mean,eda_std,temp_mean,temp_std,physicalFatigueScore,mentalFatigueScore,window_start,person,session
0,0.125039,0.080078,0.520844,0.390140,0.024092,0.001001,0.600951,0.007658,0.216412,0.010570,0.014790,0.000071,0.614152,0.139774,0.333291,0.152878,2021-08-23 08:56:41,4,1
1,0.104057,0.121734,0.507516,0.417466,0.024990,0.004274,0.600521,0.013107,0.208339,0.016107,0.014854,0.000289,0.623595,0.270942,0.333291,0.152878,2021-08-23 08:56:56,4,1
2,0.082800,0.073613,0.441414,0.372332,0.026505,0.004145,0.603213,0.014586,0.205604,0.014507,0.015288,0.001046,0.635421,0.222802,0.333291,0.152878,2021-08-23 08:57:11,4,1
3,0.083776,0.084599,0.390490,0.301961,0.026365,0.003653,0.605107,0.004934,0.211711,0.015543,0.015762,0.000342,0.644837,0.225930,0.333291,0.152878,2021-08-23 08:57:26,4,1
4,0.103404,0.099023,0.388435,0.189949,0.025492,0.006559,0.605789,0.015589,0.220641,0.015715,0.015974,0.000360,0.663780,0.606651,0.333291,0.152878,2021-08-23 08:57:41,4,1


In [ ]:
import pandas as pd

df = pd.read_csv('/content/drive/MyDrive/Fatigue_Set/final_feature_label_dataset_normalized.csv')

# Count entries per (person, session)
counts = df.groupby(['person', 'session']).size().reset_index(name='count')

print(counts)

total_counts = df.groupby('person').size().reset_index(name='total_count')

print(total_counts)

    person  session  count
0        1        1    102
1        1        2     88
2        1        3     87
3        2        1     19
4        2        2     76
5        2        3     86
6        3        1     78
7        3        2     74
8        3        3     71
9        4        1    106
10       4        2     95
11       4        3     62
12       5        1    113
13       5        2     74
14       5        3     68
15       6        1     86
16       6        2     72
17       6        3     63
18       7        1     85
19       7        2     81
20       7        3     91
21       8        1     73
22       8        2     46
23       8        3     81
24       9        1    102
25       9        2     81
26       9        3     75
27      10        1     54
28      10        2     74
29      10        3     68
30      11        1    101
31      11        2     84
32      11        3     63
33      12        1     99
34      12        2     72
35      12        3     69
 